# vLLM 推論 & チャットループ

このノートブックでは AWQ 量子化済みモデルや LoRA アダプタを読み込み、vLLM を使った推論・ベンチマーク・対話ループを個別機能として提供します。

1. 環境構築とリポジトリ clone
2. 推論設定の永続化（再起動後はこのセルまで再実行）
3. vLLM モデルのロードと LoRA 適用
4. ベンチマーク推論とメトリクス保存
5. LoRA 切り替えユーティリティ
6. チャットループ（会話ログを JSONL で保存）
7. 後片付け


In [ ]:
# === 1. Colab utility setup ===
import os
import shlex
import sys
from typing import Sequence

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover
    get_ipython = None  # type: ignore


def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython 上で実行してください。Google Colab を想定しています。")
    return ip


def clone_repo(url: str, target: str, branch: str | None = None) -> None:
    if os.path.exists(target):
        print("既存のリポジトリを再利用します:", target)
        return
    ip = _require_ipython()
    if branch:
        print(f"リポジトリを clone します: {url} (branch={branch})")
        ip.system(f"git clone --branch {branch} --single-branch {url} {target}")
    else:
        print("リポジトリを clone します:", url)
        ip.system(f"git clone {url} {target}")


def _run_pip(arguments: Sequence[str]) -> None:
    ip = _require_ipython()
    command = " ".join(shlex.quote(str(arg)) for arg in arguments)
    print("pip", command)
    ip.run_line_magic("pip", command)


def pip_install(
    packages: Sequence[str],
    *,
    index_url: str | None = None,
    extra_index_url: str | None = None,
    upgrade: bool = True,
    extra_args: Sequence[str] | None = None,
) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["install"]
    if upgrade:
        args.append("--upgrade")
    if index_url:
        args.extend(["--index-url", index_url])
    if extra_index_url:
        args.extend(["--extra-index-url", extra_index_url])
    if extra_args:
        args.extend(extra_args)
    args.extend(items)
    _run_pip(args)


def pip_uninstall(packages: Sequence[str]) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["uninstall", "-y"]
    args.extend(items)
    _run_pip(args)


REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR
REPO_BRANCH = "main2"  # 必要に応じて変更

IS_COLAB = "google.colab" in sys.modules
DEFAULT_TORCH_INDEX = "https://download.pytorch.org/whl/cu126" if IS_COLAB else None
TORCH_INDEX_URL = os.environ.get("PYTORCH_WHL_INDEX_URL", DEFAULT_TORCH_INDEX)
TORCH_EXTRA_INDEX_URL = os.environ.get("PYTORCH_EXTRA_INDEX_URL", "https://pypi.org/simple")

BASE_BOOTSTRAP = [
    "pip>=24.2",
    "setuptools==79.0.1",
    "wheel>=0.44.0",
    "packaging>=24.2",
    "jedi>=0.19.1",
]

TORCH_PACKAGES = [
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
]

VLLM_SUPPORT_PACKAGES = [
    "ninja",
    "cmake>=3.26.1",
    "numba==0.61.2",
    "llvmlite==0.44.0",
    "xformers==0.0.32.post1",
    "ray[cgraph]==2.50.1",
    "msgspec==0.19.0",
    "gguf>=0.17.1",
    "datasets>=4.0.0,<5.0.0",
]
VLLM_CORE_PACKAGES = [
    "compressed-tensors==0.11.0",
    "vllm==0.11.0",
]

EXTRA_PACKAGES: Sequence[str] = []

clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
pip_install(BASE_BOOTSTRAP)
pip_install(TORCH_PACKAGES, index_url=TORCH_INDEX_URL, extra_index_url=TORCH_EXTRA_INDEX_URL)
pip_install(VLLM_SUPPORT_PACKAGES)
pip_install(VLLM_CORE_PACKAGES, extra_args=["--no-deps"])
if EXTRA_PACKAGES:
    pip_install(EXTRA_PACKAGES)

print("依存パッケージのインストールが完了しました。必要ならランタイム再起動後にセル1→2を再実行してください。")


In [ ]:
# === 2. 推論設定と永続化 ===
import json
import os
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

COLAB_DATA_ROOT_DEFAULT = Path("/content/drive/MyDrive/ProjectForte/llm-lab-save")
COLAB_WORKSPACE_ROOT_DEFAULT = Path("/content/llm-lab-save")
COLAB_DATA_ROOT = Path(os.environ.get("LLMLAB_DRIVE_DATA_ROOT", COLAB_DATA_ROOT_DEFAULT))
workspace_override = os.environ.get("LLMLAB_WORKSPACE_DATA_ROOT")
COLAB_WORKSPACE_ROOT = Path(workspace_override) if workspace_override else COLAB_WORKSPACE_ROOT_DEFAULT

if IS_COLAB:
    from google.colab import drive  # type: ignore

    skip_mount = os.environ.get("LLMLAB_SKIP_DRIVE_MOUNT", "").lower() in {"1", "true", "yes"}
    if skip_mount:
        print("環境変数 LLMLAB_SKIP_DRIVE_MOUNT により Google Drive マウントをスキップします。")
    else:
        drive.mount("/content/drive", force_remount=False)

    if workspace_override:
        COLAB_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
    else:
        COLAB_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        if COLAB_WORKSPACE_ROOT.exists() or COLAB_WORKSPACE_ROOT.is_symlink():
            print("既存のワークスペースリンクを再利用します:", COLAB_WORKSPACE_ROOT)
        else:
            COLAB_WORKSPACE_ROOT.symlink_to(COLAB_DATA_ROOT, target_is_directory=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
else:
    local_override = os.environ.get("LLMLAB_LOCAL_DATA_ROOT")
    DATA_ROOT = Path(local_override) if local_override else Path.cwd() / "save"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path(REPO_DIR).resolve()
STATE_DIR = DATA_ROOT / "runtime_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = STATE_DIR / "inference_chat_vllm.json"

DEFAULT_CONFIG = {
    "MODEL_PATH": str(DATA_ROOT / "artifacts" / "awq_qwen3_14b_ojousama"),
    "QUANTIZATION": "awq",
    "LORA_PATH": None,
    "MERGE_LORA": False,
    "VLLM_OPTIONS": {
        "tensor_parallel_size": 1,
        "dtype": "auto",
        "max_model_len": 4096,
        "gpu_memory_utilization": 0.9,
        "download_dir": str(DATA_ROOT / "cache" / "huggingface"),
        "trust_remote_code": True,
    },
    "GENERATION": {
        "MAX_NEW_TOKENS": 512,
        "TEMPERATURE": 0.7,
        "TOP_P": 0.9,
        "REPETITION_PENALTY": 1.05,
        "STOP": ["
User:", "User:"],
    },
    "CHAT": {
        "SYSTEM_PROMPT": "You are a helpful assistant based on Qwen3-14B.",
        "STOP_PHRASES": ["/exit", ":q"],
        "EXIT_COMMAND": "/exit",
    },
    "BENCHMARK": {
        "PROMPTS": [
            "Summarise the following specification in Japanese:
- Multi-stage LoRA tuned Qwen3-14B
- Target deployment on lightweight edge devices",
            "Explain three benefits of deploying this LoRA-enhanced model as a customer-facing FAQ bot.",
        ],
        "PROMPTS_FILE": None,
        "PROMPTS_SPLIT": "validation",
        "PROMPTS_COLUMN": "text",
        "SAVE_GENERATIONS_TO": str(DATA_ROOT / "logs" / "benchmark_generations.jsonl"),
        "SAVE_METRICS_TO": str(DATA_ROOT / "logs" / "benchmark_metrics.json"),
    },
    "CHAT_LOG_PATH": str(DATA_ROOT / "logs" / "chat_history.jsonl"),
}


def _deep_merge(base: dict, override: dict) -> dict:
    result = dict(base)
    for key, value in override.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = _deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def load_state(path: Path, default: dict) -> dict:
    if not path.exists():
        return dict(default)
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(f"保存済みの設定を復元しました: {path}")
        return _deep_merge(default, payload)
    except Exception as exc:
        print(f"設定ファイルの読み込みに失敗したためデフォルトを使用します: {exc}")
        return dict(default)


def persist_config(config: dict) -> None:
    STATE_FILE.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"設定を保存しました: {STATE_FILE}")


CONFIG = load_state(STATE_FILE, DEFAULT_CONFIG)
CONFIG["MODEL_PATH"] = str(Path(CONFIG["MODEL_PATH"]))
if CONFIG.get("LORA_PATH"):
    CONFIG["LORA_PATH"] = str(Path(CONFIG["LORA_PATH"]))
CONFIG["BENCHMARK"] = _deep_merge(DEFAULT_CONFIG["BENCHMARK"], CONFIG.get("BENCHMARK", {}))
CONFIG["GENERATION"] = _deep_merge(DEFAULT_CONFIG["GENERATION"], CONFIG.get("GENERATION", {}))
CONFIG["CHAT"] = _deep_merge(DEFAULT_CONFIG["CHAT"], CONFIG.get("CHAT", {}))
CONFIG["VLLM_OPTIONS"] = _deep_merge(DEFAULT_CONFIG["VLLM_OPTIONS"], CONFIG.get("VLLM_OPTIONS", {}))
CONFIG["VLLM_OPTIONS"]["download_dir"] = str(Path(CONFIG["VLLM_OPTIONS"].get("download_dir", DATA_ROOT / "cache" / "huggingface")))

Path(CONFIG["VLLM_OPTIONS"]["download_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["MODEL_PATH"]).parent.mkdir(parents=True, exist_ok=True)
Path(CONFIG["BENCHMARK"].get("SAVE_GENERATIONS_TO", str(DATA_ROOT / "logs"))).parent.mkdir(parents=True, exist_ok=True)
Path(CONFIG["CHAT_LOG_PATH"]).parent.mkdir(parents=True, exist_ok=True)

persist_config(CONFIG)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_PATH: {CONFIG['MODEL_PATH']}")
print(f"LORA_PATH: {CONFIG.get('LORA_PATH')}")
print("設定を変更したら persist_config(CONFIG) を実行して保存してください。")


In [ ]:
# === 3. vLLM モデルロード ===
from pathlib import Path
from typing import Any, Dict, Optional

from src.llmlab.backends.vllm_backend import load_model, free_model

bundle = globals().get("bundle")


def _build_vllm_config(
    *,
    lora_path: Optional[str] = None,
    merge: Optional[bool] = None,
    quantization: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = dict(CONFIG.get("VLLM_OPTIONS", {}))
    cfg["model_name"] = CONFIG["MODEL_PATH"]
    cfg["quantization"] = quantization or CONFIG.get("QUANTIZATION", "none")
    cfg["lora_path"] = lora_path if lora_path is not None else CONFIG.get("LORA_PATH")
    cfg["merge_lora"] = bool(CONFIG.get("MERGE_LORA", False)) if merge is None else bool(merge)
    cfg.setdefault("download_dir", str(Path(CONFIG["VLLM_OPTIONS"].get("download_dir", Path(CONFIG["MODEL_PATH"]).parent))))
    cfg.setdefault("_timings", {})
    return cfg


def reload_bundle(
    *,
    lora_path: Optional[str] = None,
    merge: Optional[bool] = None,
    quantization: Optional[str] = None,
) -> None:
    global bundle
    if bundle is not None:
        free_model(bundle)
        bundle = None
    cfg = _build_vllm_config(lora_path=lora_path, merge=merge, quantization=quantization)
    cfg["_timings"] = {}
    bundle = load_model(cfg)
    CONFIG["LORA_PATH"] = cfg.get("lora_path")
    CONFIG["MERGE_LORA"] = cfg.get("merge_lora", False)
    CONFIG["QUANTIZATION"] = cfg.get("quantization", "none")
    persist_config(CONFIG)
    print("モデルをロードしました。LoRA:", CONFIG.get("LORA_PATH"), "量子化:", CONFIG.get("QUANTIZATION"))


reload_bundle()


In [ ]:
# === 4. ベンチマーク推論 ===
import json
from pathlib import Path
from typing import List

from src.llmlab.backends.vllm_backend import profile_generation

if bundle is None:
    raise RuntimeError("モデルがロードされていません。セル3を実行してください。")

bench_cfg = CONFIG.get("BENCHMARK", {})


def _collect_prompts(cfg: dict) -> List[str]:
    prompts: List[str] = [p for p in cfg.get("PROMPTS", []) if p]
    prompts_file = cfg.get("PROMPTS_FILE")
    if prompts_file:
        path = Path(prompts_file)
        if path.exists():
            prompts.extend([line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()])
        else:
            try:
                from datasets import load_dataset

                dataset = load_dataset(prompts_file, split=cfg.get("PROMPTS_SPLIT", "validation"))
                column = cfg.get("PROMPTS_COLUMN", "text")
                prompts.extend([item for item in dataset[column] if item])
            except Exception as exc:
                raise RuntimeError(f"ベンチマークデータ {prompts_file} の読み込みに失敗しました: {exc}")
    # 重複を除去しつつ順序維持
    seen = set()
    ordered: List[str] = []
    for prompt in prompts:
        if prompt not in seen:
            ordered.append(prompt)
            seen.add(prompt)
    return ordered


def _generation_kwargs() -> dict:
    gen_cfg = CONFIG.get("GENERATION", {})
    stop_list = list(gen_cfg.get("STOP", []) or [])
    chat_stop = CONFIG.get("CHAT", {}).get("STOP_PHRASES", [])
    exit_cmd = CONFIG.get("CHAT", {}).get("EXIT_COMMAND")
    for phrase in chat_stop:
        if phrase and phrase not in stop_list:
            stop_list.append(phrase)
    if exit_cmd and exit_cmd not in stop_list:
        stop_list.append(exit_cmd)
    return {
        "max_new_tokens": gen_cfg.get("MAX_NEW_TOKENS", 512),
        "temperature": gen_cfg.get("TEMPERATURE", 0.7),
        "top_p": gen_cfg.get("TOP_P", 0.9),
        "repetition_penalty": gen_cfg.get("REPETITION_PENALTY", 1.05),
        "stop": stop_list,
    }

prompts = _collect_prompts(bench_cfg)
if not prompts:
    raise ValueError("ベンチマーク用のプロンプトが設定されていません。CONFIG['BENCHMARK'] を確認してください。")

outputs, metrics = profile_generation(bundle, prompts, **_generation_kwargs())

save_generations = bench_cfg.get("SAVE_GENERATIONS_TO")
if save_generations:
    path = Path(save_generations)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fp:
        for prompt, output in zip(prompts, outputs):
            fp.write(json.dumps({"prompt": prompt, "output": output}, ensure_ascii=False) + "
")
    print("生成結果を保存しました:", path)

save_metrics = bench_cfg.get("SAVE_METRICS_TO")
if save_metrics:
    path = Path(save_metrics)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    print("メトリクスを保存しました:", path)

CONFIG["LAST_BENCHMARK"] = {
    "prompts": len(prompts),
    "generations_path": save_generations,
    "metrics_path": save_metrics,
    "metrics": metrics,
}
persist_config(CONFIG)

print("ベンチマークが完了しました。tokens_out:", metrics.get("tokens_out"))


In [ ]:
# === 5. LoRA 切り替えユーティリティ ===
from typing import Optional


def switch_lora(lora_path: Optional[str], *, merge: Optional[bool] = None, quantization: Optional[str] = None) -> None:
    """LoRA / 量子化設定を切り替えてモデルを再読み込みします。"""
    resolved = str(Path(lora_path)) if lora_path else None
    reload_bundle(lora_path=resolved, merge=merge, quantization=quantization)


print("例: switch_lora('/content/drive/MyDrive/.../adapter', merge=False)")
print("LoRA を解除する場合: switch_lora(None)")


In [ ]:
# === 6. チャットループ（ログ保存付き） ===
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List

from src.llmlab.backends.vllm_backend import generate_texts
from src.llmlab.backends.vllm_backend import _build_chat_prompt  # type: ignore

if bundle is None:
    raise RuntimeError("モデルがロードされていません。セル3を実行してください。")

chat_cfg = CONFIG.get("CHAT", {})
log_path = Path(CONFIG.get("CHAT_LOG_PATH", Path(DATA_ROOT) / "logs" / "chat_history.jsonl"))
log_path.parent.mkdir(parents=True, exist_ok=True)

exit_cmd = chat_cfg.get("EXIT_COMMAND", "/exit")
stop_phrases = set(chat_cfg.get("STOP_PHRASES", [])) | {exit_cmd}
stop_lower = {phrase.lower() for phrase in stop_phrases}
print(f"チャットを開始します。終了するには {exit_cmd} を入力してください。")

history: List[Dict[str, str]] = []
log_entries: List[Dict[str, str]] = []

try:
    while True:
        user_input = input("You> ").strip()
        if not user_input:
            continue
        if user_input.lower() in stop_lower:
            print("チャットを終了します。")
            break
        history.append({"role": "user", "content": user_input})
        log_entries.append({
            "timestamp": datetime.utcnow().isoformat(),
            "role": "user",
            "content": user_input,
        })
        prompt = _build_chat_prompt(chat_cfg.get("SYSTEM_PROMPT", ""), history)
        reply = generate_texts(bundle, [prompt], **{
            "max_new_tokens": CONFIG["GENERATION"].get("MAX_NEW_TOKENS", 512),
            "temperature": CONFIG["GENERATION"].get("TEMPERATURE", 0.7),
            "top_p": CONFIG["GENERATION"].get("TOP_P", 0.9),
            "repetition_penalty": CONFIG["GENERATION"].get("REPETITION_PENALTY", 1.05),
            "stop": list(stop_phrases),
        })[0]
        history.append({"role": "assistant", "content": reply})
        log_entries.append({
            "timestamp": datetime.utcnow().isoformat(),
            "role": "assistant",
            "content": reply,
        })
        print(f"Assistant> {reply}")
finally:
    if log_entries:
        session_id = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
        with log_path.open("a", encoding="utf-8") as fp:
            for entry in log_entries:
                payload = {
                    "session_id": session_id,
                    "model_path": CONFIG.get("MODEL_PATH"),
                    "lora_path": CONFIG.get("LORA_PATH"),
                    "quantization": CONFIG.get("QUANTIZATION"),
                    **entry,
                }
                fp.write(json.dumps(payload, ensure_ascii=False) + "
")
        print("会話ログを保存しました:", log_path)


In [ ]:
# === 7. 後片付け ===
import gc

if bundle is not None:
    free_model(bundle)
    bundle = None
    print("vLLM モデルを解放しました。")

gc.collect()
print("クリーンアップが完了しました。")
